# CLIP–Visual Jev / SNLI-VE 可复现研究

目标：E/N/C 三分类；seed=20261010；新样本，不能复原同学的 100 条。

本 notebook 嵌入了脚本。**GPU 部分尚未在交付环境实跑；无模型结果。** 需要 Flickr30K 图片。
建议支持 BF16 的 Colab L4/A100，官方依赖要求 CUDA 13.0 兼容驱动。
当前 100 条矩阵：accuracy 78%，macro-F1 77.61%；E→N 9 条。

按顺序运行；校准和 student 选择结束后才运行独立测试阶段。不要看完 test 再调参。


In [1]:
# 首次运行安装；之后“运行时 → 重启会话”，从下一单元继续。
import subprocess, sys
def install(*args): subprocess.run([sys.executable, '-m', 'pip', 'install', *args], check=True)
install('torch==2.14.0', 'torchvision==0.29.0', '--index-url', 'https://download.pytorch.org/whl/cu130')
install('transformers==5.17.0', 'peft==0.21.0', 'accelerate==1.15.0',
        'safetensors==0.8.0', 'tokenizers==0.23.2', 'huggingface_hub==1.32.0',
        'numpy==2.2.6', 'pillow==12.3.0')
print('安装完成。请重启会话，然后从下一单元继续。')


安装完成。请重启会话，然后从下一单元继续。


### flickr30k 图片集导入

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install -U huggingface_hub hf_xet

from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(
    repo_id="nlphuji/flickr30k",
    filename="flickr30k-images.zip",
    repo_type="dataset",
    local_dir="/content"
)

print("下载完成：", zip_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 40.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.17.0 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.


flickr30k-images.zip: reconstructing file:   0%|          |  0.00B / 4.39GB            

flickr30k-images.zip: downloading bytes:           |  0.00B            

下载完成： /content/flickr30k-images.zip


In [5]:
import zipfile
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

with zipfile.ZipFile(zip_path, "r") as z:
    print("压缩包内部示例：", z.namelist()[:5])
    z.extractall(drive_root)

print("解压完成")

压缩包内部示例： ['flickr30k-images/', '__MACOSX/._flickr30k-images', 'flickr30k-images/2609797461.jpg', '__MACOSX/flickr30k-images/._2609797461.jpg', 'flickr30k-images/1788892671.jpg']
解压完成


In [ ]:
IMAGES = Path("/content/drive/MyDrive/flickr30k-images")

jpg_files = list(IMAGES.glob("*.jpg"))

print("文件夹存在：", IMAGES.is_dir())
print("JPG 图片数量：", len(jpg_files))
print("示例图片：", jpg_files[:3])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json
WORK = Path('/content/drive/MyDrive/clip_jev_snli_ve_seed20261010')
IMAGES = Path('/content/drive/MyDrive/flickr30k-images')  # 改为实际图片目录，内部是 <image_id>.jpg
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
assert IMAGES.is_dir(), '先取得 Flickr30K 图片，并设置 IMAGES；本包不包含图片。'
def run(*args): subprocess.run([sys.executable, *map(str,args)], check=True)


In [ ]:
# 写出内嵌脚本；若脚本已修改则停止，避免混合实验版本。
FILES = {'study.py': '"""SNLI-VE study. Saved class order is ALWAYS E,N,C. See README_zh.md."""\nimport argparse\nimport collections\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport subprocess\nimport sys\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nLABELS = [\'E\', \'N\', \'C\']\nLABEL_MAP = {\'entailment\': 0, \'neutral\': 1, \'contradiction\': 2}\nSEED = 20261010\nJEV_COMMIT = \'a392eaedfa41adae348f9767b55c0f54b3a13b57\'\nBASE = \'Qwen/Qwen3-VL-4B-Instruct\'\nADAPTER = \'guanxuyu/visual-jev-4b-answer-sft\'\n\ndef read(path):\n    with open(path, encoding=\'utf-8\') as f:\n        return [json.loads(s) for s in f if s.strip()]\n\ndef stream(path):\n    with open(path, encoding=\'utf-8\') as f:\n        for s in f:\n            if s.strip(): yield json.loads(s)\n\ndef write(path, rows):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'w\', encoding=\'utf-8\', newline=\'\\n\') as f:\n        for row in rows:\n            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\ndef save(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef rank(*parts):\n    return hashlib.sha256(\'|\'.join(map(str, parts)).encode()).hexdigest()\n\ndef softmax(z):\n    z = np.asarray(z, dtype=np.float64)\n    z = z - z.max(axis=-1, keepdims=True)\n    p = np.exp(z)\n    return p / p.sum(axis=-1, keepdims=True)\n\ndef metrics(y, p):\n    y = np.asarray(y, dtype=int); p = np.asarray(p, dtype=float)\n    pred = p.argmax(1)\n    cm = np.zeros((3, 3), dtype=int)\n    np.add.at(cm, (y, pred), 1)\n    tp = cm.diagonal(); support = cm.sum(1); predicted = cm.sum(0)\n    precision = tp / np.maximum(predicted, 1)\n    recall = tp / np.maximum(support, 1)\n    f1 = 2 * tp / np.maximum(support + predicted, 1)\n    conf = p.max(1); correct = (pred == y)\n    ece = 0.\n    for i in range(10):\n        mask = (conf >= i / 10) & (conf < (i + 1) / 10 if i < 9 else conf <= 1)\n        if mask.any(): ece += mask.mean() * abs(conf[mask].mean() - correct[mask].mean())\n    return dict(n=len(y), accuracy=float(correct.mean()), macro_f1=float(f1.mean()),\n                balanced_accuracy=float(recall.mean()), confusion_matrix=cm.tolist(),\n                precision=precision.tolist(), recall=recall.tolist(), f1=f1.tolist(),\n                nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()),\n                brier=float(((p - np.eye(3)[y]) ** 2).sum(1).mean()), ece_10=float(ece),\n                E_to_N=int(cm[0, 1]), N_to_E=int(cm[1, 0]),\n                exact_max_ties=int(((p == p.max(1, keepdims=True)).sum(1) > 1).sum()))\n\ndef select_rows(path, split, count, seed):\n    groups = collections.defaultdict(dict)\n    ids = set(); invalid = 0; duplicate = 0; conflicts = set()\n    for r in stream(path):\n        if r.get(\'gold_label\') not in LABEL_MAP:\n            invalid += 1; continue\n        fid = str(r[\'Flickr30K_ID\']).removesuffix(\'.jpg\')\n        sid = str(r[\'pairID\']); hyp = r[\'sentence2\'].strip()\n        if sid in ids: duplicate += 1; continue\n        ids.add(sid)\n        key = hyp\n        if (fid, key) in conflicts: continue\n        candidate = dict(sample_id=sid, image_id=fid, image_file=f\'{fid}.jpg\',\n                         hypothesis=hyp, label=LABEL_MAP[r[\'gold_label\']], source_split=split)\n        if key in groups[fid]:\n            if groups[fid][key][\'label\'] != LABEL_MAP[r[\'gold_label\']]:\n                # Deterministic data-quality rule BEFORE sampling; no model outputs involved.\n                conflicts.add((fid,key)); del groups[fid][key]; continue\n            if sid < groups[fid][key][\'sample_id\']: groups[fid][key] = candidate\n            duplicate += 1; continue\n        groups[fid][key] = candidate\n    eligible = [fid for fid in groups if groups[fid]]\n    if len(eligible) < count: raise ValueError(f\'{split}: need {count} images; available {len(eligible)}\')\n    chosen = sorted(eligible, key=lambda x: rank(seed, split, x))[:count]\n    # One hypothesis per image, selected WITHOUT looking at its label.\n    rows = [min(groups[fid].values(), key=lambda r: rank(seed, r[\'sample_id\'])) for fid in chosen]\n    return rows, set(groups), dict(valid_images=len(eligible), invalid_labels=invalid, duplicates=duplicate,\n                                   excluded_conflicting_image_hypothesis_pairs=len(conflicts))\n\ndef prepare(args):\n    out = Path(args.out)\n    if (out / \'manifest_meta.json\').exists(): raise FileExistsError(\'Use a new output directory; manifests are immutable.\')\n    counts = [args.train_n, args.val_n, args.test_n]\n    all_groups = {}; summaries = {}; hashes = {}; selected = {}\n    for split, n in zip([\'train\', \'dev\', \'test\'], counts):\n        path = Path(args.annotations) / f\'snli_ve_{split}.jsonl\'\n        rows, all_groups[split], summaries[split] = select_rows(path, split, n, args.seed)\n        selected[split] = rows; hashes[split] = sha(path)\n    for a, b in [(\'train\', \'dev\'), (\'train\', \'test\'), (\'dev\', \'test\')]:\n        assert not all_groups[a] & all_groups[b], f\'Source image overlap: {a}/{b}\'\n    dev = selected.pop(\'dev\')\n    cut = len(dev) // 2\n    selected[\'val_fit\'] = dev[:cut]; selected[\'val_select\'] = dev[cut:]\n    for split, rows in selected.items():\n        for r in rows: r[\'split\'] = split\n        assert set(r[\'label\'] for r in rows) == {0, 1, 2}, f\'{split}: missing a class\'\n        write(out / f\'{split}.jsonl\', rows)\n    save(out / \'manifest_meta.json\', dict(seed=args.seed, labels=LABELS,\n        selection=\'SHA256-ranked image groups; one label-blind hypothesis per image\',\n        source_hashes=hashes, source_summary=summaries,\n        manifests={s: dict(n=len(r), counts=dict(collections.Counter(LABELS[x[\'label\']] for x in r)),\n                           sha256=sha(out / f\'{s}.jsonl\')) for s, r in selected.items()},\n        note=\'New sample; cannot reconstruct the original 100 examples.\'))\n\ndef validate_images(args):\n    from PIL import Image\n    # Reject exact pixel duplicates, including files re-encoded under different IDs.\n    seen = {}; records = []\n    for split in [\'train\', \'val_fit\', \'val_select\', \'test\']:\n        for r in read(Path(args.manifests) / f\'{split}.jsonl\'):\n            path = Path(args.images) / r[\'image_file\']\n            with Image.open(path) as im:\n                im = im.convert(\'RGB\')\n                pixel_sha = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()\n            if pixel_sha in seen:\n                raise ValueError(f\'Duplicate pixels: {seen[pixel_sha]} and {r["sample_id"]}. Do not silently resample.\')\n            seen[pixel_sha] = r[\'sample_id\']\n            records.append(dict(sample_id=r[\'sample_id\'], split=split, pixel_sha256=pixel_sha, file_sha256=sha(path)))\n    write(Path(args.manifests) / \'image_audit.jsonl\', records)\n\ndef provenance():\n    import importlib.metadata as md\n    return {name: md.version(name) for name in [\'torch\', \'transformers\', \'peft\', \'numpy\', \'huggingface_hub\']}\n\ndef infer(args):\n    import torch\n    from PIL import Image\n    from peft import PeftModel\n    from huggingface_hub import snapshot_download\n    if not torch.cuda.is_available(): raise RuntimeError(\'Use Colab with a CUDA GPU.\')\n    torch.manual_seed(SEED)\n    dtype = torch.bfloat16 if args.dtype == \'bfloat16\' else torch.float16\n    if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():\n        raise RuntimeError(\'BF16 unavailable. Prefer an L4/A100; FP16 is a separately named experiment.\')\n    repo = Path(args.repo).resolve()\n    actual = subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip()\n    assert actual == JEV_COMMIT, f\'Unexpected official source commit: {actual}\'\n    assert not subprocess.check_output([\'git\', \'-C\', str(repo), \'diff\', \'HEAD\', \'--\', \'code\'], text=True).strip(), \'Official code has local edits\'\n    sys.path.insert(0, str(repo / \'code\'))\n    from vdm.models.vdm_model import VDM\n    from vdm.models.prompts import prefix_text, suffix_text, CLAIM_OPTIONS\n    lock = json.loads((ROOT / \'versions.json\').read_text())\n    base_path = snapshot_download(BASE, revision=lock[\'base_sha\'])\n    adapter_path = snapshot_download(ADAPTER, revision=lock[\'adapter_sha\'])\n    config = json.loads((Path(adapter_path) / \'adapter_config.json\').read_text())\n    assert config[\'base_model_name_or_path\'] == BASE\n    assert config[\'peft_type\'] == \'LORA\'\n    model = VDM(base_path, device=\'cuda\', dtype=dtype, with_heads=False)\n    model.processor.image_processor.max_pixels = 200704\n    model.backbone = PeftModel.from_pretrained(model.backbone, adapter_path).eval()\n    model.eval()\n    assert model.heads is None\n    rows = read(args.manifest)\n    if len({r[\'sample_id\'] for r in rows}) != len(rows): raise ValueError(\'Duplicate IDs\')\n    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)\n    audit_path = Path(args.manifest).parent / \'image_audit.jsonl\'\n    audit = {r[\'sample_id\']: r for r in read(audit_path)}\n    meta = dict(versions=provenance(), revisions=lock, dtype=args.dtype, gpu=torch.cuda.get_device_name(),\n        official_commit=actual, manifest_sha256=sha(args.manifest), image_audit_sha256=sha(audit_path), labels=LABELS,\n        option_tokens=model.option_token_ids[:3].tolist(), official_order=[\'E\',\'C\',\'N\'], saved_order=LABELS,\n        prompt_prefix=prefix_text(\'\'), suffix_example=suffix_text(rows[0][\'hypothesis\'], CLAIM_OPTIONS, \'claim\'),\n        path=\'run_independent; with_heads=False; lm_option_logits\', max_pixels=200704,\n        tie_rule=\'First maximum in saved E,N,C order; exact ties counted in reports\')\n    meta_path = out / \'run.json\'\n    if meta_path.exists():\n        assert json.loads(meta_path.read_text()) == meta, \'Resume configuration changed\'\n    else: save(meta_path, meta)\n    for name in [\'base\', \'jev\']:\n        dest = out / f\'{name}.jsonl\'\n        done = read(dest) if dest.exists() else []\n        done_ids = [r[\'sample_id\'] for r in done]\n        assert len(set(done_ids)) == len(done_ids)\n        assert set(done_ids) <= {r[\'sample_id\'] for r in rows}\n        completed = set(done_ids)\n        with dest.open(\'a\', encoding=\'utf-8\', newline=\'\\n\') as f:\n            for i, row in enumerate(rows):\n                if row[\'sample_id\'] in completed: continue\n                path = Path(args.images) / row[\'image_file\']\n                assert sha(path) == audit[row[\'sample_id\']][\'file_sha256\'], \'Image changed since audit\'\n                with Image.open(path) as im: image = im.convert(\'RGB\')\n                q = dict(qtype=\'claim\', instruction=row[\'hypothesis\'], candidates=None)\n                with torch.inference_mode():\n                    group = model.prepare_group(image, [q], shared_context=\'\')\n                    if name == \'base\':\n                        with model.backbone.disable_adapter(): result = model.run_independent(group)\n                    else: result = model.run_independent(group)\n                    logits = result[\'lm_option_logits\'][0, :3].float().cpu().numpy()[[0, 2, 1]]\n                if not np.isfinite(logits).all(): raise ValueError(\'Nonfinite logits\')\n                p = softmax(logits)\n                record = dict(row, model=name, logits=logits.tolist(), probabilities=p.tolist(),\n                    prediction=LABELS[int(p.argmax())], image_sha256=audit[row[\'sample_id\']][\'file_sha256\'])\n                f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + \'\\n\'); f.flush()\n                if i % 20 == 0: print(name, i, \'/\', len(rows), flush=True)\n    print(\'Completed paired base/adapter inference.\', flush=True)\n\ndef aligned(path, manifest=None):\n    rows = read(path)\n    ids = [r[\'sample_id\'] for r in rows]\n    assert len(ids) == len(set(ids)), \'Duplicate predictions\'\n    if manifest:\n        expected = read(manifest); by_id = {r[\'sample_id\']: r for r in rows}\n        assert set(ids) == {r[\'sample_id\'] for r in expected}, \'Missing or extra predictions\'\n        rows = [by_id[r[\'sample_id\']] for r in expected]\n        for p, r in zip(rows, expected):\n            for key in [\'label\', \'image_id\', \'hypothesis\', \'split\']: assert p[key] == r[key]\n    z = np.array([r[\'logits\'] for r in rows], float)\n    assert z.shape == (len(rows), 3) and np.isfinite(z).all()\n    return rows, np.array([r[\'label\'] for r in rows]), z\n\ndef transform(z, params):\n    return (np.asarray(z) + np.array([params[\'b\'], 0., 0.])) / params[\'T\']\n\ndef tune(zfit, yfit, zsel, ysel):\n    temperatures = np.unique(np.r_[1., np.geomspace(.25, 4., 161)])\n    def fit_t(z):\n        return float(min(temperatures, key=lambda t: metrics(yfit, softmax(z / t))[\'nll\']))\n    raw = dict(b=0., T=1.)\n    t_only = dict(b=0., T=fit_t(zfit))\n    biases = np.linspace(-2., 2., 81)\n    # Selection objective predeclared: macro-F1, then NLL, then smaller absolute bias.\n    def key(b):\n        m = metrics(ysel, softmax(zsel + [b, 0., 0.]))\n        return (-m[\'macro_f1\'], m[\'nll\'], abs(b))\n    b = float(min(biases, key=key))\n    b_only = dict(b=b, T=1.)\n    both = dict(b=b, T=fit_t(zfit + [b, 0., 0.]))\n    variants = dict(raw=raw, temperature=t_only, E_bias=b_only, E_bias_temperature=both)\n    scores = {k: metrics(ysel, softmax(transform(zsel, v))) for k, v in variants.items()}\n    selected = min(variants, key=lambda k: (-scores[k][\'macro_f1\'], scores[k][\'nll\'], abs(variants[k][\'b\'])))\n    return dict(variants=variants, selected=selected, selected_params=variants[selected], validation=scores,\n                bias_at_boundary=abs(b) >= 2., rule=\'macro-F1 then NLL; grids locked before test\')\n\ndef calibrate(args):\n    out = Path(args.out)\n    if out.exists(): raise FileExistsError(\'Calibration is locked. Use a new experiment directory.\')\n    result = dict(labels=LABELS, seed=SEED, models={}, validation_hashes={})\n    for name in [\'base\', \'jev\']:\n        paths = [Path(args.predictions) / s / f\'{name}.jsonl\' for s in [\'val_fit\',\'val_select\']]\n        a, ya, za = aligned(paths[0], Path(args.manifests) / \'val_fit.jsonl\')\n        b, yb, zb = aligned(paths[1], Path(args.manifests) / \'val_select.jsonl\')\n        assert not {r[\'image_id\'] for r in a} & {r[\'image_id\'] for r in b}\n        result[\'models\'][name] = tune(za, ya, zb, yb)\n        result[\'validation_hashes\'][name] = [sha(p) for p in paths]\n        errors = []\n        for row, z in zip(b, zb):\n            if row[\'label\'] == 0 and z.argmax() == 1:\n                errors.append(dict(row, margin_N_minus_E=float(z[1]-z[0]),\n                    minimum_E_bias_to_win=float(max(z[1], z[2])-z[0]),\n                    review_category=\'\', reviewer_note=\'\'))\n        write(out.parent / f\'{name}_validation_E_to_N.jsonl\', errors)\n    save(out, result)\n\ndef bootstrap_delta(y, pa, pb, image_ids, seed=SEED, repeats=2000):\n    # Cluster bootstrap remains valid if future manifests retain several hypotheses per image.\n    groups = collections.defaultdict(list)\n    for i, key in enumerate(image_ids): groups[key].append(i)\n    groups = list(groups.values()); rng = np.random.default_rng(seed); values = []\n    for _ in range(repeats):\n        idx = np.concatenate([groups[j] for j in rng.integers(len(groups), size=len(groups))])\n        a = metrics(y[idx], pa[idx]); b = metrics(y[idx], pb[idx])\n        values.append([b[\'accuracy\']-a[\'accuracy\'], b[\'macro_f1\']-a[\'macro_f1\']])\n    ci = np.quantile(values, [.025,.975], axis=0)\n    return dict(delta_accuracy_95ci=ci[:,0].tolist(), delta_macro_f1_95ci=ci[:,1].tolist())\n\ndef evaluate(args):\n    out = Path(args.out)\n    if out.exists(): raise FileExistsError(\'Test report already exists: do not tune after opening it.\')\n    cal = json.loads(Path(args.calibration).read_text())\n    report = dict(labels=LABELS, calibration_sha256=sha(args.calibration), scores={}, paired={}, prediction_hashes={})\n    probabilities = {}; reference = None\n    for name in [\'base\',\'jev\']:\n        path = Path(args.predictions) / \'test\' / f\'{name}.jsonl\'\n        rows, y, z = aligned(path, Path(args.manifests) / \'test.jsonl\')\n        report[\'prediction_hashes\'][name] = sha(path)\n        if reference is None: reference = rows\n        assert [r[\'sample_id\'] for r in rows] == [r[\'sample_id\'] for r in reference]\n        variants = cal[\'models\'][name][\'variants\']\n        for variant, params in variants.items():\n            p = softmax(transform(z, params)); key = f\'{name}/{variant}\'\n            probabilities[key] = p; report[\'scores\'][key] = metrics(y, p)\n        chosen = cal[\'models\'][name][\'selected\']\n        p = probabilities[f\'{name}/{chosen}\']\n        write(out.parent / f\'{name}_test_calibrated.jsonl\', [dict(r,\n              calibrated_logits=transform(zz, variants[chosen]).tolist(), calibrated_probabilities=pp.tolist(),\n              calibrated_prediction=LABELS[int(pp.argmax())], calibration_variant=chosen) for r, zz, pp in zip(rows,z,p)])\n    ids = [r[\'image_id\'] for r in reference]\n    pairs = [(\'base/raw\',\'jev/raw\'), (\'jev/raw\',\'jev/\'+cal[\'models\'][\'jev\'][\'selected\'])]\n    for a,b in pairs:\n        report[\'paired\'][f\'{b} minus {a}\'] = bootstrap_delta(y, probabilities[a], probabilities[b], ids)\n    save(out, report)\n\ndef main():\n    p = argparse.ArgumentParser(); sub = p.add_subparsers(dest=\'command\', required=True)\n    q = sub.add_parser(\'prepare\'); q.add_argument(\'--annotations\', required=True); q.add_argument(\'--out\', default=\'manifests\')\n    q.add_argument(\'--seed\', type=int, default=SEED); q.add_argument(\'--train-n\', type=int, default=6000)\n    q.add_argument(\'--val-n\', type=int, default=800); q.add_argument(\'--test-n\', type=int, default=1000)\n    q.set_defaults(func=prepare)\n    q = sub.add_parser(\'audit-images\'); q.add_argument(\'--images\', required=True); q.add_argument(\'--manifests\', default=\'manifests\'); q.set_defaults(func=validate_images)\n    q = sub.add_parser(\'infer\'); q.add_argument(\'--manifest\', required=True); q.add_argument(\'--images\', required=True)\n    q.add_argument(\'--repo\', default=\'Visual-Jev\'); q.add_argument(\'--out\', required=True)\n    q.add_argument(\'--dtype\', choices=[\'bfloat16\',\'float16\'], default=\'bfloat16\'); q.set_defaults(func=infer)\n    q = sub.add_parser(\'calibrate\'); q.add_argument(\'--predictions\', default=\'predictions\'); q.add_argument(\'--manifests\', default=\'manifests\')\n    q.add_argument(\'--out\', default=\'results/calibration.json\'); q.set_defaults(func=calibrate)\n    q = sub.add_parser(\'evaluate\'); q.add_argument(\'--predictions\', default=\'predictions\'); q.add_argument(\'--manifests\', default=\'manifests\')\n    q.add_argument(\'--calibration\', default=\'results/calibration.json\'); q.add_argument(\'--out\', default=\'results/test_report.json\'); q.set_defaults(func=evaluate)\n    args = p.parse_args(); args.func(args)\n\nif __name__ == \'__main__\': main()\n', 'student.py': '"""Frozen CLIP pair features + learned E/N/C head: CE vs raw/calibrated teacher KD."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport random\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom study import read, write, save, sha, aligned, softmax, metrics, transform, bootstrap_delta, SEED\n\nCLIP_ID = \'openai/clip-vit-base-patch32\'\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\ndef features(args):\n    from PIL import Image\n    from transformers import CLIPModel, CLIPProcessor\n    from huggingface_hub import HfApi\n    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)\n    lock = out / \'clip_revision.json\'\n    if not lock.exists(): save(lock, dict(model=CLIP_ID, revision=HfApi().model_info(CLIP_ID).sha))\n    revision = json.loads(lock.read_text())[\'revision\']\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    model = CLIPModel.from_pretrained(CLIP_ID, revision=revision).to(device).eval()\n    processor = CLIPProcessor.from_pretrained(CLIP_ID, revision=revision)\n    audit = {r[\'sample_id\']: r for r in read(Path(args.manifests) / \'image_audit.jsonl\')}\n    for split in args.splits:\n        manifest = Path(args.manifests) / f\'{split}.jsonl\'\n        dest = out / f\'{split}.npz\'\n        if dest.exists(): raise FileExistsError(dest)\n        rows = read(manifest); values = []; truncated = []\n        for start in range(0, len(rows), 32):\n            batch = rows[start:start+32]; images = []; texts = []\n            for row in batch:\n                path = Path(args.images) / row[\'image_file\']\n                assert sha(path) == audit[row[\'sample_id\']][\'file_sha256\']\n                with Image.open(path) as im: images.append(im.convert(\'RGB\'))\n                texts.append(row[\'hypothesis\'])\n                if len(processor.tokenizer(row[\'hypothesis\'])[\'input_ids\']) > 77:\n                    truncated.append(row[\'sample_id\'])\n            inputs = processor(text=texts, images=images, padding=True, truncation=True,\n                               max_length=77, return_tensors=\'pt\').to(device)\n            with torch.inference_mode():\n                output = model(**inputs)\n                v = F.normalize(output.image_embeds.float(), dim=-1)\n                t = F.normalize(output.text_embeds.float(), dim=-1)\n                x = torch.cat([v, t, torch.abs(v-t), v*t], dim=-1)\n            values.append(x.cpu().numpy())\n        np.savez_compressed(dest, x=np.concatenate(values), y=[r[\'label\'] for r in rows],\n                            ids=[r[\'sample_id\'] for r in rows], image_ids=[r[\'image_id\'] for r in rows])\n        save(out / f\'{split}_meta.json\', dict(manifest_sha256=sha(manifest), revision=revision,\n              truncated_ids=truncated, feature_order=[\'v\',\'t\',\'abs(v-t)\',\'v*t\'], frozen_encoder=True))\n\ndef load_features(directory, split, manifests):\n    path = Path(directory) / f\'{split}.npz\'\n    data = np.load(path, allow_pickle=False)\n    rows = read(Path(manifests) / f\'{split}.jsonl\')\n    assert data[\'ids\'].tolist() == [r[\'sample_id\'] for r in rows]\n    assert data[\'y\'].tolist() == [r[\'label\'] for r in rows]\n    meta = json.loads((Path(directory) / f\'{split}_meta.json\').read_text())\n    assert meta[\'manifest_sha256\'] == sha(Path(manifests) / f\'{split}.jsonl\')\n    return data\n\ndef head(dim):\n    return nn.Sequential(nn.Linear(dim, 512), nn.ReLU(), nn.Dropout(.1), nn.Linear(512, 3))\n\ndef train(args):\n    out = Path(args.out)\n    if out.exists(): raise FileExistsError(\'Student training output exists. Use a new experiment directory.\')\n    out.mkdir(parents=True)\n    a = load_features(args.features, \'train\', args.manifests)\n    b = load_features(args.features, \'val_select\', args.manifests)\n    _, yteacher, zteacher = aligned(Path(args.predictions)/\'train\'/\'jev.jsonl\', Path(args.manifests)/\'train.jsonl\')\n    assert np.array_equal(yteacher, a[\'y\'])\n    cal = json.loads(Path(args.calibration).read_text())[\'models\'][\'jev\'][\'selected_params\']\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    x = torch.tensor(a[\'x\'], device=device); y = torch.tensor(a[\'y\'], dtype=torch.long, device=device)\n    xv = torch.tensor(b[\'x\'], device=device)\n    # Teacher outputs only on TRAIN; validation teacher predictions never become training targets.\n    targets = {\'CE\': None, \'KD_raw\': zteacher, \'KD_calibrated\': transform(zteacher, cal)}\n    configs = []; history = []\n    for seed in args.seeds:\n        for method, z in targets.items():\n            seed_all(seed)\n            model = head(x.shape[1]).to(device)\n            opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)\n            qt = None if z is None else torch.tensor(softmax(z/2.), dtype=torch.float32, device=device)\n            best = (-1., -float(\'inf\')); best_epoch = None\n            for epoch in range(1, args.epochs+1):\n                model.train(); total = 0.\n                for idx in torch.randperm(len(y), device=device).split(128):\n                    zs = model(x[idx]); ce = F.cross_entropy(zs, y[idx])\n                    loss = ce if qt is None else .5*ce + .5*4.*F.kl_div(\n                        F.log_softmax(zs/2., dim=-1), qt[idx], reduction=\'batchmean\')\n                    opt.zero_grad(); loss.backward(); opt.step(); total += loss.item()*len(idx)\n                model.eval()\n                with torch.inference_mode(): p = model(xv).softmax(-1).cpu().numpy()\n                m = metrics(b[\'y\'],p); key = (m[\'macro_f1\'], -m[\'nll\'])\n                history.append(dict(seed=seed,method=method,epoch=epoch,loss=total/len(y),validation=m))\n                if key > best:\n                    best = key; best_epoch = epoch\n                    torch.save(model.cpu().state_dict(), out/f\'{method}_{seed}.pt\'); model.to(device)\n            configs.append(dict(method=method,seed=seed,epoch=best_epoch,validation_macro_f1=best[0],\n                                checkpoint=f\'{method}_{seed}.pt\', dimension=int(x.shape[1])))\n    save(out/\'selection.json\', dict(configs=configs, alpha=.5, tau=2., epochs=args.epochs,\n          calibration_sha256=sha(args.calibration), teacher_train_sha256=sha(Path(args.predictions)/\'train\'/\'jev.jsonl\'),\n          feature_hashes={s:sha(Path(args.features)/f\'{s}.npz\') for s in [\'train\',\'val_select\']},\n          note=\'Frozen CLIP; trainable pair classifier. No backbone fine-tuning.\'))\n    write(out/\'history.jsonl\',history)\n\ndef evaluate(args):\n    out=Path(args.out)\n    if out.exists(): raise FileExistsError(\'Student test report already exists.\')\n    data=load_features(args.features,\'test\',args.manifests)\n    selected=json.loads((Path(args.students)/\'selection.json\').read_text())\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    x=torch.tensor(data[\'x\'],device=device); rows=read(Path(args.manifests)/\'test.jsonl\')\n    results={}; probs={}\n    for c in selected[\'configs\']:\n        model=head(c[\'dimension\']).to(device)\n        model.load_state_dict(torch.load(Path(args.students)/c[\'checkpoint\'],map_location=device,weights_only=True)); model.eval()\n        with torch.inference_mode(): z=model(x).cpu().numpy()\n        p=softmax(z); key=f\'{c["method"]}_{c["seed"]}\'; probs[key]=p; results[key]=metrics(data[\'y\'],p)\n        write(out.parent/f\'{key}_test.jsonl\',[dict(r,logits=zz.tolist(),probabilities=pp.tolist()) for r,zz,pp in zip(rows,z,p)])\n    paired={}\n    for seed in sorted({c[\'seed\'] for c in selected[\'configs\']}):\n        for method in [\'KD_raw\',\'KD_calibrated\']:\n            paired[f\'{method}_minus_CE_seed_{seed}\']=bootstrap_delta(data[\'y\'],probs[f\'CE_{seed}\'],probs[f\'{method}_{seed}\'],data[\'image_ids\'])\n    averages={method:float(np.mean([v[\'macro_f1\'] for k,v in results.items() if k.startswith(method+\'_\')]))\n              for method in [\'CE\',\'KD_raw\',\'KD_calibrated\']}\n    save(out,dict(scores=results,paired=paired,mean_macro_f1=averages,selection_sha256=sha(Path(args.students)/\'selection.json\')))\n\ndef main():\n    p=argparse.ArgumentParser(); sub=p.add_subparsers(dest=\'command\',required=True)\n    q=sub.add_parser(\'features\'); q.add_argument(\'--images\',required=True); q.add_argument(\'--manifests\',default=\'manifests\')\n    q.add_argument(\'--out\',default=\'features\'); q.add_argument(\'--splits\',nargs=\'+\',default=[\'train\',\'val_select\']); q.set_defaults(func=features)\n    q=sub.add_parser(\'train\'); q.add_argument(\'--features\',default=\'features\'); q.add_argument(\'--manifests\',default=\'manifests\')\n    q.add_argument(\'--predictions\',default=\'predictions\'); q.add_argument(\'--calibration\',default=\'results/calibration.json\')\n    q.add_argument(\'--out\',default=\'students\'); q.add_argument(\'--epochs\',type=int,default=30)\n    q.add_argument(\'--seeds\',nargs=\'+\',type=int,default=[20261010,20261011,20261012]); q.set_defaults(func=train)\n    q=sub.add_parser(\'evaluate\'); q.add_argument(\'--features\',default=\'features\'); q.add_argument(\'--manifests\',default=\'manifests\')\n    q.add_argument(\'--students\',default=\'students\'); q.add_argument(\'--out\',default=\'results/student_test_report.json\'); q.set_defaults(func=evaluate)\n    args=p.parse_args(); args.func(args)\n\nif __name__==\'__main__\': main()\n', 'download_annotations.py': '"""Download version-pinned annotation files only; Flickr30K images are supplied separately."""\nimport json\nfrom pathlib import Path\nimport urllib.request\nROOT = Path(__file__).resolve().parent\nversion = json.loads((ROOT / \'versions.json\').read_text())[\'dataset_sha\']\nout = ROOT / \'annotations\'; out.mkdir(exist_ok=True)\nfor split in [\'train\', \'dev\', \'test\']:\n    name = f\'snli_ve_{split}.jsonl\'; dest = out / name\n    if dest.exists():\n        print(\'Already downloaded:\', dest, flush=True); continue\n    url = f\'https://huggingface.co/datasets/HuggingFaceM4/SNLI-VE/resolve/{version}/{name}\'\n    print(\'Downloading\', name, flush=True)\n    part = dest.with_suffix(\'.partial\')\n    with urllib.request.urlopen(url, timeout=180) as response, part.open(\'wb\') as f:\n        while chunk := response.read(1024 * 1024): f.write(chunk)\n    part.replace(dest)\n    print(dest, dest.stat().st_size, flush=True)\n', 'test_study.py': "import argparse\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nimport numpy as np\nimport study as s\n\nclass StudyTests(unittest.TestCase):\n    def test_confusion_matrix(self):\n        cm = [[31,9,4],[3,20,3],[0,3,27]]; y=[]; p=[]\n        for i,row in enumerate(cm):\n            for j,n in enumerate(row):\n                y.extend([i]*n); p.extend([np.eye(3)[j]]*n)\n        m=s.metrics(y,p)\n        self.assertEqual(m['confusion_matrix'],cm)\n        self.assertAlmostEqual(m['accuracy'],.78)\n        self.assertEqual(m['E_to_N'],9)\n        self.assertAlmostEqual(m['recall'][0],31/44)\n\n    def test_temperature_cannot_change_argmax(self):\n        z=np.random.default_rng(1).normal(size=(100,3))\n        for t in [.01,.5,1,5,100]:\n            np.testing.assert_array_equal(z.argmax(1),s.softmax(z/t).argmax(1))\n\n    def test_bias_and_label_order(self):\n        official=np.array([[2.,1.,3.]]) # E,C,N\n        enc=official[:,[0,2,1]]\n        self.assertEqual(s.LABELS[enc.argmax()], 'N')\n        corrected=s.transform(enc,dict(b=1.1,T=2.))\n        self.assertEqual(s.LABELS[corrected.argmax()], 'E')\n        np.testing.assert_allclose(s.softmax(corrected).sum(1),[1.])\n\n    def test_prepare_reproducible_and_isolated(self):\n        with tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent) as d:\n            root=Path(d)\n            for split in ['train','dev','test']:\n                rows=[]\n                for i in range(60):\n                    for k in range(2):\n                        rows.append(dict(Flickr30K_ID=f'{split}{i}',pairID=f'{split}{i}_{k}',\n                                         sentence2=f'hypothesis {i} {k}',gold_label=list(s.LABEL_MAP)[i%3]))\n                s.write(root/f'snli_ve_{split}.jsonl',rows)\n            def run(out):\n                s.prepare(argparse.Namespace(annotations=str(root),out=str(root/out),seed=42,train_n=60,val_n=60,test_n=60))\n            run('a'); run('b')\n            for split in ['train','val_fit','val_select','test']:\n                self.assertEqual(s.sha(root/'a'/f'{split}.jsonl'),s.sha(root/'b'/f'{split}.jsonl'))\n            a=s.read(root/'a/val_fit.jsonl'); b=s.read(root/'a/val_select.jsonl')\n            self.assertFalse({r['image_id'] for r in a}&{r['image_id'] for r in b})\n            with self.assertRaises(FileExistsError): run('a')\n\n    def test_alignment_reorders_and_rejects_missing(self):\n        with tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent) as d:\n            d=Path(d)\n            rows=[dict(sample_id=str(i),image_id=str(i),hypothesis='h',split='val_fit',label=i,logits=[0.,1.,2.]) for i in range(3)]\n            s.write(d/'manifest',rows); s.write(d/'pred',rows[::-1])\n            actual,_,_=s.aligned(d/'pred',d/'manifest')\n            self.assertEqual([r['sample_id'] for r in actual],['0','1','2'])\n            s.write(d/'pred',rows[:2])\n            with self.assertRaises(AssertionError): s.aligned(d/'pred',d/'manifest')\n\n    def test_calibration_uses_valid_parameters(self):\n        y=np.tile(np.arange(3),20)\n        z=np.eye(3)[y]*2; z[:,0]-=2.3\n        result=s.tune(z,y,z,y)\n        self.assertGreater(result['variants']['E_bias']['b'],0)\n        self.assertGreater(result['selected_params']['T'],0)\n        self.assertGreaterEqual(result['validation'][result['selected']]['macro_f1'],result['validation']['raw']['macro_f1'])\n\n    def test_paired_bootstrap_identity(self):\n        y=np.tile(np.arange(3),10); p=np.eye(3)[y]*.8+.2/3\n        result=s.bootstrap_delta(y,p,p,[str(i//2) for i in range(30)],repeats=30)\n        self.assertEqual(result['delta_accuracy_95ci'],[0.,0.])\n        self.assertEqual(result['delta_macro_f1_95ci'],[0.,0.])\n\n    def test_conflict_exclusion_is_order_independent(self):\n        with tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent) as d:\n            path=Path(d)/'rows'\n            rows=[dict(Flickr30K_ID='1',pairID='a',sentence2='conflict',gold_label='entailment'),\n                  dict(Flickr30K_ID='1',pairID='b',sentence2='conflict',gold_label='neutral'),\n                  dict(Flickr30K_ID='1',pairID='c',sentence2='valid',gold_label='entailment'),\n                  dict(Flickr30K_ID='1',pairID='d',sentence2='valid',gold_label='entailment')]\n            s.write(path,rows); first,_,summary=s.select_rows(path,'train',1,42)\n            s.write(path,rows[::-1]); second,_,_=s.select_rows(path,'train',1,42)\n            self.assertEqual(first,second); self.assertEqual(first[0]['sample_id'],'c')\n            self.assertEqual(summary['excluded_conflicting_image_hypothesis_pairs'],1)\n\nif __name__=='__main__': unittest.main()\n", 'versions.json': '{\n  "base_sha": "ebb281ec70b05090aa6165b016eac8ec08e71b17",\n  "adapter_sha": "7a3f1bb0d8f7bfbbd8514d968ab7de3518849c8d",\n  "dataset_sha": "54e07e370ffb19c1b5eebf13055e8b6af33991ad",\n  "jev_commit": "a392eaedfa41adae348f9767b55c0f54b3a13b57"\n}\n', 'README_zh.md': "# CLIP–Visual Jev：SNLI-VE 三分类研究包\n\n研究目标是提高基于 CLIP 的 SNLI-VE E/N/C 分类。Amazon 不作为主要目标。本包不使用 Qwen2.5，不引入自造 q/r、充分性头或所谓内部置信机制。\n\n## 已有 100 条混淆矩阵\n\n用户提供的数字，按行真实标签、按列预测标签，顺序都是 E/N/C：\n\n|真实\\预测|E|N|C|合计|\n|---|---:|---:|---:|---:|\n|E|31|9|4|44|\n|N|3|20|3|26|\n|C|0|3|27|30|\n\n- Accuracy = 78/100 = **78.00%**；macro-F1 ≈ **77.61%**；balanced accuracy ≈ **79.13%**。\n- E：precision 91.18%，recall 70.45%，F1 79.49%。\n- N：precision 62.50%，recall 76.92%，F1 68.97%。\n- C：precision 79.41%，recall 90.00%，F1 84.38%。\n- E→N 为 9/44=20.45% 的真实 E，9/22=40.91% 的全部错误。N→E 为 3/26=11.54%。预测分布 34/32/34，真实分布 44/26/30。\n- 这支持“检查 E→N”的优先级，不能证明错误全来自固定 E 偏置、模型能力或标签问题。缺少图像、假设、逐条输出，不能判断这 9 条的语义原因、logit margin 或 bias 可修复率。\n- 若 100 条独立同分布，准确率 Wilson 95% 区间约 68.9%–85.0%；抽样方式和图像相关性未知，因此只作示意，不作模型优劣检验。\n\n**不能复原同学的原样本。** 重新抽样、校准的结果不能解释为对原 100 条逐条纠错，也不能与原矩阵作配对显著性检验。\n\n## 已核实的推理接口\n\n[适配器模型卡](https://huggingface.co/guanxuyu/visual-jev-4b-answer-sft)及下载的 adapter_config.json 均指向 `Qwen/Qwen3-VL-4B-Instruct`。本包固定模型、适配器、注释 SHA，以及官方代码 commit，详见 versions.json。配置文件记录 LoRA r=16、alpha=32、语言侧目标模块和 visual 排除规则；本包只使用已发布 adapter，不加载额外 decision heads。\n\n直接调用[官方代码](https://github.com/guanxuyu-sv/Visual-Jev/tree/a392eaedfa41adae348f9767b55c0f54b3a13b57/code)：`VDM(..., with_heads=False)`、`prepare_group(qtype='claim')`、`run_independent`、`lm_option_logits`。使用同一模型和提示，通过 PEFT 的 `disable_adapter()` 获取无适配器基线，再启用适配器获取 Jev。两者都 eval，输入相同，分辨率上限 200704 pixels；模型加载失败会中止，不会退回其他模型。\n\n官方 prompts.py 的选项顺序是 supported / contradicted / not determined，即 **E/C/N**；读出是 `Answer:` 最后位置的下一 token ` A`、` B`、` C` logits，官方会断言候选字符串各为单 token。保存前显式重排 `[0,2,1]`，因此所有输出数组都是 **E/N/C**。不是直接抽取词汇 E/N/C 的 logits，也不是自由生成后解析。\n\nsoftmax 只在这三个候选分数上归一化，是条件于选项集合的分布，不是已校准的客观证据概率。这里保留官方 SNLI-VE claim 提示和空 shared_context，绝不输入 sentence1 文本 premise 或金标签。这里是新样本实验，不声称复现模型卡分数。\n\n精确相等的最大分数按保存顺序 E/N/C 取首个类别，并在报告记录 tie 数量；N/C 完全平局时，这与官方 E/C/N 原顺序取首项存在差异。不要把零 margin 平局当作可靠类别偏好。\n\n## 数据与独立性\n\n[SNLI-VE 原仓库](https://github.com/necla-ml/SNLI-VE)规定官方划分图像不相交；[HF 注释镜像](https://huggingface.co/datasets/HuggingFaceM4/SNLI-VE)提供 JSONL，图片需另行获得 Flickr30K。download_annotations.py 只下载固定版本注释，不执行远程 dataset 脚本。\n\n默认 seed=20261010：官方 train 选 6000 张图，dev 选 800 张图，test 选 1000 张图；每张图再按固定哈希顺序选一条假设，不按类别挑选，保留过滤后图像均匀抽样的自然类别分布（不等同于原始逐行数据分布）。所有候选先按 image ID 分组，跨官方集合检查重叠；同图相同假设去重，同标签时取字典序最小 pairID；若标签冲突，在抽样前排除整个同图同假设组。使用 seed 加 SHA256 排序，避免源文件顺序和不同随机库版本改变样本。dev 分成 val_fit 400 与 val_select 400。manifest 保存 pairID、图像 ID、hypothesis、标签及来源划分，元数据保存注释哈希和类别计数。\n\n已实际生成的样本分布如下，顺序 E/N/C：train **2020/1970/2010**；val_fit **109/152/139**；val_select **138/132/130**；test **316/327/357**。全量注释 train/dev/test 分别排除 **150/5/4** 个冲突组。所有选定图像 ID 相互独立；尚无原图，像素级审计需在 Colab 补做。这个经过统一清理和每图抽一条的测试子集不是官方全量 benchmark，报告时注明区别。\n\n图像到位后，audit-images 检查每个文件，并对 RGB 像素算 SHA256，拒绝换名/重编码的完全相同图像。这个检查不保证排除视觉近重复；如发现重复，必须在任何推理前制定排除规则并新建实验，不能按结果挑替代样本。\n\n独立测试的含义是：不参与本研究的校准、训练、checkpoint 选择。发布适配器本来用过 SNLI-VE train，且作者用过 SNLI-VE test 做评估；官方 build_other.py 明确从该 test 构建评估数据。因此不能将本研究测试称为对作者模型开发完全未暴露，更不能排除 Qwen/CLIP 预训练污染。用于比较已发布模型及本地改进是合理的，但不足以宣称全新域泛化。\n\n## 校准与 E→N 分析\n\n设原始分数为 z=(zE,zN,zC)，使用 **softmax((z + [bE,0,0])/T)**。正温度不改变 argmax，只影响 NLL/Brier/ECE；bias 才改变分类。\n\n预先固定：T 在 [0.25,4] 的 161 点对数网格，另含 T=1；bE 在 [-2,2] 间隔 0.05。val_fit 最小化 NLL 拟合 T；val_select 以 macro-F1 为主、NLL 为次选择 bE 和方案，允许负 bias 和不调整。输出 raw、temperature、E_bias、E_bias_temperature 四个消融。选择边界值会记录，不在看过测试后扩网格。\n\n锁定 calibration.json 后才执行测试。测试输出 accuracy、macro-F1、balanced accuracy、各类 P/R/F1、混淆矩阵、NLL、Brier、10-bin ECE，以及按图像配对 bootstrap 95% 区间。代码拒绝覆写已有测试报告，但这不是防止人为重复调参的安全机制；研究者需遵守冻结协议。小验证集 ECE 波动较大。\n\n验证集 E→N 清单保存 margin_N_minus_E 和使 E 获胜所需的最小 bias（相等是平局，实际需略大）。人工核查图像和假设，记录：对象/动作漏检、属性/数量/关系、不可见意图或背景知识、标注疑义、预处理/截断问题。这里的分类是待检验假设，不是已发现的错误原因。不得用测试错误决定规则或重标主要测试集；原标签指标保留，标签复核另列。\n\n## 蒸馏方案与已实现范围\n\n使用 `openai/clip-vit-base-patch32` 冻结编码器，以归一化 image/text 特征构成 `[v,t,|v-t|,v*t]`，训练 512 维隐藏层的三分类 MLP。CLIP 原始相似度本身不是 E/N/C 分类器。\n\n三组严格共用样本、初始化种子、优化器与训练预算：CE 真标签基线；CE+原始 Jev 蒸馏；CE+验证集所选校准 Jev 蒸馏。损失为 `(1-alpha) CE + alpha*tau^2 KL(p_teacher_tau || p_student_tau)`，alpha=0.5、tau=2。校准温度 T 与蒸馏温度 tau 分开，软目标由校准 logits/tau 得到。只用 train 软目标，保留真实标签。30 epochs、AdamW lr=1e-3、weight_decay=1e-4、batch=128；按 val_select macro-F1/NLL 选 checkpoint，种子 20261010/11/12。\n\n这一步蒸馏到“基于冻结 CLIP 的三分类系统”，**没有更新 CLIP 编码器本身**。它首先检验 teacher 软分布是否带来收益。后续若扩展到 CLIP 编码器微调，可解冻最后 1–2 层、使用更小学习率，并保持 CE 与 KD 同等训练能力。必须预先重新分配评估集或用新独立 holdout；不要看完当前测试后反复改善再称为一次性测试。若校准 KD 没有超过 CE，报告失败而不是预设 Jev 必须优于基础模型。\n\n## Colab 操作\n\n1. 将 `CLIP_Visual_Jev_SNLI_VE.ipynb` 上传到 Colab。它嵌入全部必要脚本，无需上传源码目录。选择支持 BF16 的 GPU（L4/A100）；显存占用尚未实测。\n2. 安装单元固定官方记录的软件版本。它使用官方 CUDA 13.0 PyTorch 构建，要求 Colab 驱动兼容；若安装或 CUDA 检查失败，先换兼容运行环境。不要静默换版本后混合结果。安装完成后重启会话，再从路径设置单元继续。\n3. 挂载 Drive，设置持久化 `WORK` 和已合法获取的 `flickr30k-images` 目录 `IMAGES`。图片不包含在交付物中；参照 SNLI-VE 原仓库下载说明。HF 需要认证时通过 Colab secrets/HF 登录配置，不在 notebook 写 token。\n4. Notebook 写出代码，下载固定版本注释、生成清单、审计图片。若已有清单，则复用；不要为了得到好分数换种子。首次调通可另建 smoke 目录，使用少量图像，但不能把 smoke 当正式结果。\n5. 先对 train/val_fit/val_select 生成 base 与 Jev logits；支持按 sample_id 续跑，恢复时检查配置一致性。推理逐条写入并 flush，意外断线造成最后一行损坏时备份后仅移除不完整尾行，不删除已完成记录。\n6. 冻结校准；提取 CLIP train/val 特征，训练三种 student。每个阶段使用独立进程，释放教师 GPU 内存。\n7. 最后生成 test logits 与 CLIP test 特征，输出两个测试报告。已完成结果不能覆写。下载或保留 results、predictions、manifests、features/clip_revision.json、students/selection.json 和环境记录。\n\n命令行等价入口：`python study.py --help`、`python student.py --help`。若换成 FP16/T4，显式传 `--dtype float16`，单独命名实验目录；它不是模型卡 BF16 条件的等价复现。不要随意降低图像分辨率或使用量化来混称同一设置。\n\n## 验证范围与交付状态\n\n本地无 PyTorch/CUDA 模型运行环境，未运行 Qwen 权重、适配器推理或 CLIP 训练；没有生成任何真实模型性能提升结论。已经实际获取并核实官方推理代码、模型配置、版本 ID，下载三个官方划分的固定版本注释并生成 7800 条样本清单。CPU 单元检查覆盖矩阵计算、E/C/N→E/N/C 映射、温度不改变决策、bias 方向、抽样可复现与隔离、冲突标签剔除、缺失预测拒绝、校准及配对 bootstrap。GPU 路径仍须在 Colab 完成端到端验收。\n\n来源：以上接口依据固定版本的官方代码与模型卡；[Qwen3-VL 模型卡](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct)；[CLIP Transformers 文档](https://huggingface.co/docs/transformers/model_doc/clip)。依赖版本来自官方 code/requirements.txt；本包的抽样、校准、student 是本研究新增方法，不冒充官方算法。\n", 'expected_manifest_meta.json': '{\n  "seed": 20261010,\n  "labels": [\n    "E",\n    "N",\n    "C"\n  ],\n  "selection": "SHA256-ranked image groups; one label-blind hypothesis per image",\n  "source_hashes": {\n    "train": "aaa48e40a6ca002db7f09cf57cee117198f91afd232b57e789ad196bb8dcbc9c",\n    "dev": "25aac5515c68293fd879f69951f14da407495e9decf630ab8dfbd77b64ce0cfd",\n    "test": "3fffd289d99097df26d9855126f51b23c5c5a62f6c30dc275b9faaa2f50fabb8"\n  },\n  "source_summary": {\n    "train": {\n      "valid_images": 29783,\n      "invalid_labels": 0,\n      "duplicates": 1189,\n      "excluded_conflicting_image_hypothesis_pairs": 150\n    },\n    "dev": {\n      "valid_images": 1000,\n      "invalid_labels": 0,\n      "duplicates": 31,\n      "excluded_conflicting_image_hypothesis_pairs": 5\n    },\n    "test": {\n      "valid_images": 1000,\n      "invalid_labels": 0,\n      "duplicates": 34,\n      "excluded_conflicting_image_hypothesis_pairs": 4\n    }\n  },\n  "manifests": {\n    "train": {\n      "n": 6000,\n      "counts": {\n        "C": 2010,\n        "E": 2020,\n        "N": 1970\n      },\n      "sha256": "9dc440880cda17e1332630b703263068b468912034e164a07e41c074d1d56f85"\n    },\n    "test": {\n      "n": 1000,\n      "counts": {\n        "N": 327,\n        "C": 357,\n        "E": 316\n      },\n      "sha256": "a5dbd16bad3e17bc35dcef326dd0f70ee82ffc1a6b9298c4a60e8d820e4ac257"\n    },\n    "val_fit": {\n      "n": 400,\n      "counts": {\n        "N": 152,\n        "E": 109,\n        "C": 139\n      },\n      "sha256": "a58e2f00047f159ca7b6415564ff03b699c68897fea805fbfb9088729fdbbaf4"\n    },\n    "val_select": {\n      "n": 400,\n      "counts": {\n        "C": 130,\n        "N": 132,\n        "E": 138\n      },\n      "sha256": "da47eb2435d159d7e9343314c593ec4e0775b34d7a337c472b2d35b240b251d2"\n    }\n  },\n  "note": "New sample; cannot reconstruct the original 100 examples."\n}'}
for name, source in FILES.items():
    path = WORK/name
    if path.exists():
        assert path.read_text(encoding='utf-8') == source, f'{name} 已修改；请使用新的 WORK 目录。'
    else: path.write_text(source, encoding='utf-8')
print('脚本已写出。完整方法与限制见 README_zh.md。')


In [ ]:
REPO = WORK/'Visual-Jev'
COMMIT = json.loads((WORK/'versions.json').read_text())['jev_commit']
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/guanxuyu-sv/Visual-Jev.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == COMMIT
import torch
assert torch.cuda.is_available(), '需要 CUDA GPU 和兼容驱动。'
assert torch.cuda.is_bf16_supported(), '主实验要求 BF16；请选择 L4/A100 等支持的 GPU。'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())
(WORK/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
run('-m','unittest','-v','test_study.py')


## 固定抽样与图片审计
首次下载注释约 520 MB；图片另备。训练 6000、验证 400+400、测试 1000 条，每图一条。
测试标签只在此阶段用于清单及完整性检查，不用于训练或参数选择。不要展示测试预测直至冻结方案。


In [ ]:
run('download_annotations.py')
if not (WORK/'manifests/manifest_meta.json').exists():
    run('study.py','prepare','--annotations',WORK/'annotations','--out',WORK/'manifests')
from study import sha
expected=json.loads((WORK/'expected_manifest_meta.json').read_text())
actual=json.loads((WORK/'manifests/manifest_meta.json').read_text())
assert actual == expected, '抽样结果与交付的固定清单不同。停止检查数据和脚本版本。'
for split,item in expected['manifests'].items():
    assert sha(WORK/'manifests'/f'{split}.jsonl') == item['sha256']
run('study.py','audit-images','--manifests',WORK/'manifests','--images',IMAGES)
print((WORK/'manifests/manifest_meta.json').read_text())


## 教师推理（同样本、同提示、同精度）
逐条保存原始 logits 与 E/N/C softmax。每个 split 分别跑 base 与 Jev；中断后可续跑。
模型首次下载需要磁盘与时间。本单元结束后进程释放 GPU。


In [ ]:
for split in ['train','val_fit','val_select']:
    run('study.py','infer','--manifest',WORK/'manifests'/f'{split}.jsonl',
        '--images',IMAGES,'--repo',REPO,'--out',WORK/'predictions'/split)


## 仅验证集校准并冻结
val_fit 拟合温度，val_select 选择 E bias 与方案。人工只检查 validation_E_to_N 文件。
正温度不能改变类别；校准参数选好后不可根据测试重选。


In [ ]:
if not (WORK/'results/calibration.json').exists():
    run('study.py','calibrate')
calibration=json.loads((WORK/'results/calibration.json').read_text())
for name,c in calibration['models'].items():
    print(name,c['selected'],c['selected_params'],c['validation'])


## CLIP student：CE / KD_raw / KD_calibrated
冻结 CLIP，训练相同图文三分类 MLP。三种方法相同超参、相同三个种子，各自只用验证集选 epoch。
默认 30 epochs；测试前固定，不允许依据测试结果改训练预算。CLIP 77-token 截断 ID 会记录。


In [ ]:
missing=[s for s in ['train','val_select'] if not (WORK/'features'/f'{s}.npz').exists()]
if missing:
    run('student.py','features','--images',IMAGES,'--splits',*missing)
if not (WORK/'students/selection.json').exists():
    run('student.py','train')
print((WORK/'students/selection.json').read_text())


## 独立测试：仅在上述选择完成后执行一次
代码检查 calibration.json 和 student selection 均存在。报告保留 raw/T/bias/组合的预注册消融，
主结果是验证阶段选定方案。bootstrap 为同图配对差值区间。作者已用此官方 test 评估过发布模型；
独立性仅相对于本次训练和调参，不能证明作者开发或预训练未见过。


In [ ]:
assert (WORK/'results/calibration.json').exists()
assert (WORK/'students/selection.json').exists()
run('study.py','infer','--manifest',WORK/'manifests/test.jsonl','--images',IMAGES,
    '--repo',REPO,'--out',WORK/'predictions/test')
if not (WORK/'features/test.npz').exists():
    run('student.py','features','--images',IMAGES,'--splits','test')
if not (WORK/'results/test_report.json').exists(): run('study.py','evaluate')
if not (WORK/'results/student_test_report.json').exists(): run('student.py','evaluate')
print((WORK/'results/test_report.json').read_text())
print((WORK/'results/student_test_report.json').read_text())


In [ ]:
# 打包轻量研究记录；大图片、模型权重、特征缓存、原始全量注释不打包。
import zipfile
with zipfile.ZipFile(WORK/'experiment_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['manifests','predictions','results','students']:
        for path in (WORK/folder).rglob('*'):
            if path.is_file(): z.write(path,path.relative_to(WORK))
    for name in [*FILES,'environment.txt','features/clip_revision.json']:
        path=WORK/name
        if path.is_file(): z.write(path,name)
print('结果已保存在 Drive：',WORK/'experiment_results.zip')
